In [22]:
# imports needed for this project
import os
import time
import pandas as pd
import spacy
from spacy.matcher import Matcher
from classes import AccidentData, MetaData
from scrapers import read_links_from_file, scrape_metadata, write_metadata_to_csv
import re
from transformers import pipeline


In [26]:
# loading spaCy model
nlp = spacy.load('en_core_web_sm')
matcher = Matcher(nlp.vocab)

# Hugging Face zero-shot classification pipeline initzialization
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# vehicle type dictionary
vehicle_mapping = {
    "bus": "Bus", "car": "Car", "noah": "Noah", "human hauler": "Human hauler",
    "trolley": "Trolley", "chander gari": "Chander Gari", "auto rickshaw": "Auto Rickshaw",
    "cng": "CNG", "easy-bike": "Easy-bike", "truck": "Truck", "garbage truck": "Garbage Truck",
    "trailer": "Trailer", "motorcycle": "Motorcycle", "microbus": "Microbus", "scooter": "Scooter",
    "construction vehicle": "Construction vehicle", "bicycle": "Bicycle", "ambulance": "Ambulance",
    "pickup": "Pickup", "lorry": "Lorry", "paddy cutter vehicles": "Paddy cutter vehicles",
    "bulkhead": "Bulkhead", "crane": "Crane", "wrecker": "Wrecker", "tractor": "Tractor",
    "cart": "Cart", "leguna": "Leguna", "nosimon": "Nosimon", "three-wheeler": "Three-Wheeler",
    "four-wheeler": "Four-Wheeler", "votvoti": "Votvoti", "kariman": "Kariman", "mahindra": "Mahindra",
    "van": "Van", "rickshaw": "Rickshaw", "boat": "Boat", "trawler": "Trawler", "vessel": "Vessel",
    "launch": "Launch", "tanker": "Tanker", "oil tanker": "Oil Tanker", "road roller": "Road roller",
    "power tiller": "Power Tiller", "excavator": "Excavator", "train": "Train", "airplane": "Airplane",
    "pedestrian": "Pedestrian"
}

# function created for mapping vehicle names to their standard forms
def map_vehicle_names(vehicles):
    mapped_vehicles = []
    for vehicle in vehicles:
        vehicle_lower = vehicle.lower().strip()
        if vehicle_lower in vehicle_mapping:
            mapped_vehicles.append(vehicle_mapping[vehicle_lower])
        else:
            mapped_vehicles.append("Other")  # or use "Unknown" or "Null" if preferred
    return list(set(mapped_vehicles))  # removing duplicates

# adding vehicle pattern to matcher
vehicle_patterns = [[{"LEMMA": {"IN": list(vehicle_mapping.keys())}}]]
matcher.add("VEHICLE_TYPE", vehicle_patterns)

# function created for cleaning and deduplicating lists
def clean_and_deduplicate(items):
    seen = set()
    cleaned = []
    for item in items:
        singular = item.lower().rstrip('s')
        if singular not in seen:
            seen.add(singular)
            cleaned.append(item)  # just to keep the original form
    return cleaned

# function created for removing duplicates and filtering non-Bangladeshi locations
def clean_locations(locations):
    cleaned = []
    for loc in locations:
        if loc not in cleaned and loc != 'Saudi Arabia' and loc != 'UAE':
            cleaned.append(loc)
    return cleaned

# function created for cleaning and deduplicating dates and removing ages
def clean_dates(dates):
    cleaned = []
    for date in dates:
        if not date.isdigit():  # for check - if the date is purely numeric or not
            cleaned.append(date)
    return list(dict.fromkeys(cleaned))

# function created for extracting casualties
def extract_casualties(doc):
    casualties = []
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            casualties.append(ent.text)
    return clean_and_deduplicate(casualties)

# function created for extracting ages
def extract_ages(doc):
    ages = []
    for ent in doc.ents:
        if ent.label_ == "DATE" and ent.text.isdigit() and 1 <= len(ent.text) <= 2:
            ages.append(ent.text)
    return list(set(ages))  # removing duplicates if occur

# function created for extracting injured persons
def extract_injured(doc):
    injured_count = 0
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            injured_count += 1
    # pattern for "number of people INJURED" phrases or similar one 
    injured_pattern = [
        {"LIKE_NUM": True},
        {"LOWER": {"IN": ["others", "people"]}},
        {"LOWER": {"IN": ["injured", "hurt"]}}
    ]
    matcher.add("INJURED_COUNT", [injured_pattern])
    matches = matcher(doc)
    for match_id, start, end in matches:
        count_text = doc[start:end].text
        match = re.search(r'\d+', count_text)
        if match:
            count = int(match.group())
            injured_count += count
    return injured_count

# function created for extracting reason for the accident using zero-shot classification
def extract_reason_hf(text):
    candidate_labels = [
        "reckless driving", "mechanical failure", "weather conditions", "driver fatigue",
        "drunk driving", "overspeeding", "road conditions", "pedestrian error",
        "animal crossing", "poor lighting", "other"
    ]
    if len(text) > 512:
        text = text[:512]
    results = classifier(text, candidate_labels)
    top_reason = results['labels'][0]
    return top_reason

# function created for extracting sequence of actions using SpaCy (and remove duplicates also)
def extract_sequence_of_actions_spacy(doc):
    actions = []
    for sent in doc.sents:
        for token in sent:
            if token.dep_ in ('nsubj', 'ROOT', 'dobj') and token.pos_ == 'VERB':
                actions.append(token.text)
    actions = list(dict.fromkeys(actions))  # removing duplicates
    return ' '.join(actions)

# function created for extracting details
def extract_details(meta_data_list):
    processed_data = []

    for text in meta_data_list:
        doc = nlp(text.raw_text)

        # location, date, and time
        locations = [ent.text for ent in doc.ents if ent.label_ == 'GPE']
        locations = clean_locations(locations)
        date_info = [ent.text for ent in doc.ents if ent.label_ == 'DATE' or ent.label_ == 'TIME']
        date_info = clean_dates(date_info)

        # vehicles
        matches = matcher(doc)
        vehicles = [doc[start:end].text for match_id, start, end in matches]
        vehicles = map_vehicle_names(clean_and_deduplicate(vehicles))

        # casualties and their ages
        casualties = extract_casualties(doc)

        # number of injured persons
        injured = extract_injured(doc)

        # reason for the accident
        reason = extract_reason_hf(text.raw_text)

        # sequence of actions
        actions = extract_sequence_of_actions_spacy(doc)

        # AccidentData object
        accident_data = AccidentData(
            location= locations,
            date= date_info,
            vehicles= vehicles,
            casualties= casualties,
            casualties_age= extract_ages(doc),
            injured= injured,
            accident_reason= reason,
            action_sequence= actions,
            link= text.link
        )

        processed_data.append(vars(accident_data))

    return processed_data

# function created for processing and saving the data
def process_and_save(meta_data_list, output_file):
    processed_data = extract_details(meta_data_list)
    df = pd.DataFrame(processed_data)
    df.to_csv(output_file, sep=';', index=False)

links_to_scrape = read_links_from_file("test.txt")
meta_data_list = scrape_metadata(links_to_scrape)
write_metadata_to_csv(meta_data_list, "test0")
process_and_save(meta_data_list, 'processed_events.csv')

Scrapping [https://www.unb.com.bd/category/Bangladesh/man-killed-in-kushtia-road-crash/4366]
Scrapping [https://www.unb.com.bd/category/Bangladesh/truck-ambulance-collision-leaves-one-dead-in-natore/3597]
Scrapping [https://www.unb.com.bd/category/Bangladesh/lyricist-omar-faruk-dies-in-narsingdi-road-crash/104135]
Scrapping [https://www.unb.com.bd/category/Bangladesh/three-of-a-family-killed-in-kurigram-road-crash/24449]
Scrapping [https://www.unb.com.bd/category/Bangladesh/2-killed-in-chattogram-road-crash/19204]
Scrapping [https://www.unb.com.bd/category/Bangladesh/motorcyclist-killed-38-rmg-workers-hurt-in-manikganj-road-crash/2658]
Scrapping [https://www.unb.com.bd/category/Bangladesh/four-die-in-horrific-road-crash-in-gazipur/102533]
Scrapping [https://www.unb.com.bd/category/Bangladesh/25-medical-students-hurt-in-cumilla-road-crash/8067]
Scrapping [https://www.unb.com.bd/category/Bangladesh/2-motorcyclists-killed-in-narsingdi-road-accident/12404]
Scrapping [https://www.unb.com.bd